# 01 - EDA: HCRL Car-Hacking Dataset

**Input:** `data/interim/hcrl/*.parquet` (run `python scripts/build_dataset.py --dataset hcrl` first).

**Questions this notebook answers (feeds Phase 2 feature design and the report's dataset section):**
1. Class distribution per file and overall (how imbalanced?)
2. Message rate (frames/sec) over time - do attacks change the bus load?
3. Top CAN IDs per class - DoS should be dominated by ID 0x000
4. Inter-arrival time histograms per class - DoS/Fuzzy should be much shorter
5. Payload byte behaviour of injected vs normal frames

EDA is exploration only: no thresholds or features are fitted here.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from canids.utils.config import load_config, resolve_path

cfg = load_config("data")
interim = resolve_path(cfg["paths"]["interim"]) / "hcrl"
files = sorted(interim.glob("*.parquet"))
assert files, f"No parquet in {interim} - run scripts/build_dataset.py first"
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
df.info(memory_usage="deep")

## 1. Class distribution

In [ ]:
counts = df.groupby(["source_file", "label"], observed=True).size().unstack(fill_value=0)
counts["attack_share_%"] = 100 * (1 - counts.get("Normal", 0) / counts.sum(axis=1))
display(counts)
px.bar(df["label"].value_counts().reset_index(), x="label", y="count", log_y=True,
       title="HCRL frames per class (log scale)")

## 2. Frames per second over time (per file)

In [ ]:
def rate_per_second(g: pd.DataFrame) -> pd.DataFrame:
    sec = np.floor(g["timestamp"] - g["timestamp"].min()).astype(int)
    out = g.assign(second=sec).groupby(["second", "label"], observed=True).size()
    return out.rename("frames").reset_index()

for name, g in df.groupby("source_file", observed=True):
    r = rate_per_second(g)
    px.area(r, x="second", y="frames", color="label", title=f"{name}: frames/sec").show()

## 3. Top CAN IDs per class

In [ ]:
top = (df.groupby("label", observed=True)["can_id"]
         .apply(lambda s: s.value_counts().head(10))
         .rename("frames").reset_index().rename(columns={"level_1": "can_id"}))
top["can_id_hex"] = top["can_id"].map(lambda v: f"0x{v:03X}")
top

## 4. Inter-arrival time per class
Global inter-arrival = time since the previous frame in the same file (sorted by time).

In [ ]:
df = df.sort_values(["source_file", "timestamp"], kind="stable")
df["iat_ms"] = df.groupby("source_file", observed=True)["timestamp"].diff() * 1000
display(df.groupby("label", observed=True)["iat_ms"].describe(percentiles=[.05, .5, .95]))
sample = df.dropna(subset=["iat_ms"]).groupby("label", observed=True).sample(n=50_000, replace=True, random_state=42)
px.histogram(sample, x="iat_ms", color="label", nbins=200, log_y=True, barmode="overlay",
             range_x=[0, 2], title="Inter-arrival time (ms) by class")

## 5. Payload bytes: injected vs normal

In [ ]:
byte_cols = [f"b{i}" for i in range(8)]
df.groupby("label", observed=True)[byte_cols].agg(lambda s: s[s >= 0].nunique())

## Findings (fill in from the real outputs above - no invented numbers)
- Class balance: [TBD]
- Rate change during DoS: [TBD]
- Dominant IDs per attack: [TBD]
- Inter-arrival differences: [TBD]
- Implications for Phase 2 features: [TBD]